In [1]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "duckdb",
#     "numpy",
#     "pandas",
#     "polars",
#     "pyarrow",
# ]
#
# [tool.uv]
# exclude-newer = "2025-08-29T14:06:16.861197162+02:00"
# ///

# Create sample from processed and labeled flows

- should be evenly distributed over time (each month)
- should be evenly distributed over labels
    - only consider labels benign and anomalous
    - for each taxonomy in sampling timeframe evenly distributed
    - i.e. n samples for benign and each taxonomy per month

In [2]:
import duckdb
import numpy as np
import pandas as pd
import polars as pl
from pathlib import Path

In [3]:
# Show all rows and columns in the DataFrame
pl.Config.set_tbl_rows(-1)   # -1 = no row limit
pl.Config.set_tbl_cols(-1)   # -1 = no column limit
pl.Config.set_tbl_width_chars(None);  # unlimited width

In [4]:
data_path = Path("../data/flows/v1.1")
data_key = "year=*/month=*/day=*/flows.parquet"

n = 100          # samples per taxonomy (anomalous) and per month for benign
seed = 42        # change for a different deterministic sample

sample_path = Path("../data/samples/v1.1")

In [5]:
conn = duckdb.connect(":memory:")
conn.sql(f"""
    CREATE OR REPLACE VIEW flows AS
    FROM read_parquet(
        '{data_path}/{data_key}',
        hive_partitioning=True);
""");

In [6]:
year_moth_combinations = conn.sql("""
    SELECT DISTINCT year, month
    FROM flows
    ORDER BY year, month
""").fetchall()

In [7]:
year, month = year_moth_combinations[0]

In [8]:
out_path = sample_path / f"year={year}" / f"month={month}" / f"n={n}" / f"seed={seed}"
out_path.mkdir(parents=True, exist_ok=True)

In [9]:
%%time
query = f"""
    WITH base AS (
        SELECT
            *,
            COALESCE(taxonomy,'benign') AS taxonomy_norm
        FROM flows
        WHERE year = '{year}'
            AND month = '{month}'
            AND Label IN ('benign','anomalous')
    ),
    ranked AS (
        SELECT
            b.*,
            ROW_NUMBER() OVER (
                PARTITION BY taxonomy_norm
                ORDER BY hash(
                    {seed},
                    taxonomy_norm,
                    COALESCE(CAST(anomaly_id AS VARCHAR), '')
                )
            ) AS rn
        FROM base b
    ),
    sampled AS (
        SELECT * EXCLUDE (rn)
        FROM ranked
        WHERE rn <= {n}
    )
    SELECT *
    FROM sampled
    ORDER BY Timestamp
"""

conn.execute(f"""
    COPY (
    {query}
    ) TO '{out_path}/sample.parquet' (
        FORMAT PARQUET,
        COMPRESSION ZSTD,
        OVERWRITE_OR_IGNORE
    );
""");

CPU times: user 4min 37s, sys: 26min 36s, total: 31min 14s
Wall time: 1min 13s


In [10]:
%%time
df = pl.read_parquet(sample_path, hive_partitioning=True, missing_columns='insert')

CPU times: user 205 ms, sys: 3.65 s, total: 3.86 s
Wall time: 1.55 s


In [11]:
taxonomy_counts = (
    df
    .group_by(["year", "month", "taxonomy_norm"])
    .len()
    .rename({"len": "count"})
    .sort(["year", "month", "taxonomy_norm"])
    .pivot(
        values="count",
        index=["year", "month"],
        on="taxonomy_norm"
    )
    .sort(["year", "month"])
)
taxonomy_counts = taxonomy_counts.select(
    ["year", "month", "benign"] +
    sorted([
        c for c in taxonomy_counts.columns
        if c not in ("year", "month", "benign")
    ])
)

taxonomy_counts

year,month,benign,DDoSIC,alphfl,alphflHTTP,heavy_hitter,icmp_error,ipv4gretun,malphfl,mptmp,mptp,mptpHTTP,mptpla,mptplaHTTP,ntscACK,ntscICec,ntscSYNt,ntscSYNt139445,ntscSYNtSAbado,ntscTCPICdurp,ntscTCPICterp,ntscTCPRSTACKrp,ntscUDP,ntscUDPICdurp,ntscUDPUDPrp,point_to_point,ptmp,ptmpHTTP,ptmpla,ptmplaHTTP,ptpposcaUDP,salphfl,sntscSYNt
i64,str,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
2007,"""01""",100,100,100,100,100,100,34,20,100,100,100,100,100,100,100,100,100,100,100,4,100,100,100,100,100,100,100,100,100,100,100,100


In [12]:
(taxonomy_counts
    .unpivot(index=["year", "month"], variable_name="taxonomy_norm", value_name="count")
    .filter(pl.col("count") < n)
    .sort(["year", "month", "taxonomy_norm"]))

year,month,taxonomy_norm,count
i64,str,str,u32
2007,"""01""","""ipv4gretun""",34
2007,"""01""","""malphfl""",20
2007,"""01""","""ntscTCPICterp""",4
